# A Multidimensional Predictive and Clustering Approach to Understanding State-Level Chronic Disease Disparities

## Team 8 - CMPE 255 Spring 2026

**Team Members:**
- Bharath Kumar A [018221268]
- Mithil Sri Sai Devisetty [019153394]
- Mani Mokshith Noonety [019100458]
- Tej Kiran Yenugunti [019104878]

---

## Project Overview

This notebook implements a comprehensive data-driven pipeline to analyze chronic disease disparities across U.S. states. We integrate eight public health datasets and apply machine learning techniques to identify predictive factors and state archetypes.

## 1. Setup and Library Installation

In [ ]:
# Install gdown for Google Drive data downloads
!pip install gdown --quiet

## 2. Import Required Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data downloading
import gdown

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Advanced ML
from xgboost import XGBRegressor

# Statistical analysis
from scipy import stats

# Settings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## 3. Data Loading

We will load eight state-level datasets from public repositories.

### 3.1 Dataset 1: Chronic Disease (CDC BRFSS)

Contains state-level chronic disease metrics by year, topic, and gender.

In [ ]:
# Download Chronic Disease Dataset
print("Downloading Chronic Disease dataset...")
gdown_id = "1GA_camLQGvBYLLnPa9vcNQvfZHWBWJhN"
gdown.download(id=gdown_id, output="../data/ChronicDisease.csv", quiet=False)

# Load the dataset
df_chronic = pd.read_csv("../data/ChronicDisease.csv")

print(f"\n✓ Loaded Chronic Disease dataset: {df_chronic.shape}")
print(f"Columns: {df_chronic.shape[1]}")
print(f"Rows: {df_chronic.shape[0]}")

In [ ]:
# Initial exploration
print("Dataset Info:")
df_chronic.info()

print("\nFirst 5 rows:")
df_chronic.head()

In [ ]:
# Check for missing values
print("Missing values per column:")
df_chronic.isnull().sum()

### 3.2 Clean Chronic Disease Dataset

We'll clean the dataset by:
1. Removing unnecessary columns
2. Filtering relevant disease topics
3. Removing non-state entries (US aggregates, territories)
4. Handling missing values

In [ ]:
# Step 1: Drop unnecessary columns
cols_to_drop = [
    "Response", "DataValueAlt", "DataValueFootnoteSymbol", "DataValueFootnote",
    "LowConfidenceLimit", "HighConfidenceLimit",
    "StratificationCategory2", "Stratification2",
    "StratificationCategory3", "Stratification3",
    "StratificationCategoryID2", "StratificationID2",
    "StratificationCategoryID3", "StratificationID3",
    "ResponseID", "Geolocation", "LocationID", "TopicID", "QuestionID", "DataValueTypeID"
]

df_chronic_clean = df_chronic.drop(columns=[c for c in cols_to_drop if c in df_chronic.columns])

print(f"✓ Dropped {len([c for c in cols_to_drop if c in df_chronic.columns])} unnecessary columns")
print(f"Remaining columns: {df_chronic_clean.shape[1]}")

In [ ]:
# Step 2: Filter relevant disease topics (remove non-chronic disease topics)
topics_to_drop = [
    "Nutrition, Physical Activity, and Weight Status",
    "Health Status",
    "Social Determinants of Health",
    "Immunization",
    "Cognitive Health and Caregiving",
    "Maternal Health"
]

df_chronic_clean = df_chronic_clean[~df_chronic_clean["Topic"].isin(topics_to_drop)]

print(f"✓ Filtered to chronic disease topics only")
print(f"Remaining rows: {df_chronic_clean.shape[0]}")

In [ ]:
# Step 3: Remove rows with missing DataValue
df_chronic_clean = df_chronic_clean[df_chronic_clean["DataValue"].notna()]

print(f"✓ Removed rows with missing disease values")
print(f"Remaining rows: {df_chronic_clean.shape[0]}")

In [ ]:
# Step 4: Standardize data types and rename columns
df_chronic_clean["YearStart"] = df_chronic_clean["YearStart"].astype(int)
df_chronic_clean["YearEnd"] = df_chronic_clean["YearEnd"].astype(int)
df_chronic_clean["DataValue"] = pd.to_numeric(df_chronic_clean["DataValue"], errors="coerce")
df_chronic_clean["Year"] = df_chronic_clean["YearStart"]
df_chronic_clean = df_chronic_clean.rename(columns={"LocationDesc": "State"})

print("✓ Standardized data types and renamed columns")
print(f"\\nData types:")\nprint(df_chronic_clean.dtypes)

In [ ]:
# Step 5: Remove non-state entries (US aggregates, territories)
df_chronic_clean = df_chronic_clean[
    (df_chronic_clean["State"] != "United States") & 
    (df_chronic_clean["State"] != "Puerto Rico")
].reset_index(drop=True)

print(f"✓ Removed non-state entries")
print(f"Final cleaned dataset shape: {df_chronic_clean.shape}")
print(f"\\nUnique states: {df_chronic_clean['State'].nunique()}")
print(f"Unique disease topics: {df_chronic_clean['Topic'].nunique()}")

In [ ]:
# Display cleaned data sample
print("Cleaned Chronic Disease Dataset Sample:")
df_chronic_clean.head(10)

## 4. Healthcare Expenditure Data (2015-2020)

Loading state-level healthcare spending data to analyze resource allocation patterns.

In [ ]:
# Download Healthcare Expenditure Dataset
print("Downloading Healthcare Expenditure dataset...")
gdown.download(id="1fcDO9LkieCtt5C1jr5ejdHe9_MsFR0pU", output="../data/Healthexpense.csv", quiet=False)

# Load dataset
df_expenditure = pd.read_csv("../data/Healthexpense.csv")

print(f"\n✓ Loaded Healthcare Expenditure dataset: {df_expenditure.shape}")
print(f"Years covered: 2015-2020")
df_expenditure.head()

In [ ]:
# Check data structure
print("Expenditure dataset info:")
df_expenditure.info()

print("\nColumn names:")
print(df_expenditure.columns.tolist())

In [ ]:
# Calculate total expenditure 2015-2020
# Note: Need to identify year columns and aggregate
print("TODO: Aggregate spending across 2015-2020")
print("TODO: Handle missing values")
print("TODO: Standardize state names for merging")

# Initial aggregation attempt
if 'Y2015' in df_expenditure.columns:
    year_cols = ['Y2015', 'Y2016', 'Y2017', 'Y2018', 'Y2019', 'Y2020']
    df_expenditure['Total_Expense_2015_2020'] = df_expenditure[year_cols].sum(axis=1)
    print("\n✓ Calculated total expenditure")
else:
    print("\n⚠ Need to identify year columns correctly")

## 5. Healthcare Workforce Data

Loading physicians and hospitals datasets to analyze healthcare capacity.

In [ ]:
# Download Physicians Dataset
print("Downloading Physicians dataset...")
gdown.download(id="1giEZL5n4DP3G4WaVt-v7_rqdKO6m2dNa", output="../data/Physicians.csv", quiet=False)

df_physicians = pd.read_csv("../data/Physicians.csv")

print(f"\n✓ Loaded Physicians dataset: {df_physicians.shape}")
df_physicians.head()

In [ ]:
# Download Hospitals Dataset
print("Downloading Hospitals dataset...")
gdown.download(id="1X0TXbHc-5y1_W5mPSZM7SXgSVin1dSo9", output="../data/hospitals.csv", quiet=False)

df_hospitals = pd.read_csv("../data/hospitals.csv")

print(f"\n✓ Loaded Hospitals dataset: {df_hospitals.shape}")
df_hospitals.head()

In [ ]:
# Check data structure for both datasets
print("Physicians columns:", df_physicians.columns.tolist())
print("\nHospitals columns:", df_hospitals.columns.tolist())

# TODO: Calculate per 100k metrics once we have population data
print("\n⚠ Note: Will calculate Physicians_Per100k and Hospitals_Per100k after loading population data")

## 6. Environmental and Behavioral Risk Factors

Loading physical inactivity and PM2.5 pollution datasets.

In [ ]:
# Download Physical Inactivity Dataset
print("Downloading Physical Inactivity dataset...")
gdown.download(id="17CymbIRG6qrmcL-ZMYDqWTOB0c8PtaT4", output="../data/PhysicalActivity.csv", quiet=False)

df_inactivity = pd.read_csv("../data/PhysicalActivity.csv")

print(f"\n✓ Loaded Physical Inactivity dataset: {df_inactivity.shape}")
df_inactivity.head()

In [ ]:
# Download PM2.5 Pollution Dataset
print("Downloading PM2.5 Pollution dataset...")
gdown.download(id="1C1GpHWB8wjriN13Jw5O7NM8nqQOVN1AU", output="../data/pollution.csv", quiet=False)

df_pollution = pd.read_csv("../data/pollution.csv")

print(f"\n✓ Loaded PM2.5 Pollution dataset: {df_pollution.shape}")
df_pollution.head()

In [ ]:
# Download Population Dataset (needed for per-capita calculations)
print("Downloading Population dataset...")
gdown.download(id="1SiHB7Td9piKz3QpTQnDuEWCU7K6U-r6i", output="../data/Population.csv", quiet=False)

df_population = pd.read_csv("../data/Population.csv")

print(f"\n✓ Loaded Population dataset: {df_population.shape}")
df_population.head()

## 7. Summary and Next Steps

**Data Loading Complete:**
- ✅ Chronic Disease Dataset (cleaned)
- ✅ Healthcare Expenditure Dataset
- ✅ Physicians Dataset  
- ✅ Hospitals Dataset
- ✅ Physical Inactivity Dataset
- ✅ PM2.5 Pollution Dataset
- ✅ Population Dataset

**Next Steps:**
- Merge all datasets into master dataframe
- Engineer per-capita and per-100k features
- Conduct comprehensive EDA
- Build predictive ML models
- Perform clustering analysis
- Answer research questions

## 8. Per Capita Healthcare Expenditure Dataset

Loading per capita spending data for additional spending analysis.

---

**Next Steps:**
- Load remaining 7 datasets
- Clean and preprocess each dataset
- Create master dataframe
- Perform EDA
- Build ML models

In [ ]:
# Download Per Capita Expenditure Dataset
print("Downloading Per Capita Expenditure dataset...")
gdown.download(id="1duB3gZoyUW7NcpJ39r2vXiMAB7r6XjSy", output="../data/US_PER_CAPITA20.CSV", quiet=False)

df_percapita = pd.read_csv("../data/US_PER_CAPITA20.CSV")

print(f"\n✓ Loaded Per Capita Expenditure dataset: {df_percapita.shape}")
df_percapita.head()

## 9. Data Cleaning - Healthcare Expenditure Dataset

**Lead: Mithil Sri Sai Devisetty**

Cleaning and processing healthcare spending data to prepare for master dataframe integration.

In [ ]:
# Clean expenditure dataset - standardize state names and calculate totals
print("Cleaning Healthcare Expenditure dataset...")

# Check column names
print(f"Columns: {df_expenditure.columns.tolist()}")

# If State column exists, standardize it
if 'State' in df_expenditure.columns:
    df_expenditure['State'] = df_expenditure['State'].str.strip()
    print(f"✓ Standardized state names")
else:
    print("⚠ Need to identify state column name")

# Calculate total expenditure across years if not already done
year_cols = ['Y2015', 'Y2016', 'Y2017', 'Y2018', 'Y2019', 'Y2020']
if all(col in df_expenditure.columns for col in year_cols):
    df_expenditure['Total_Expense_2015_2020'] = df_expenditure[year_cols].sum(axis=1)
    print(f"✓ Calculated Total_Expense_2015_2020")
    print(f"Mean total expenditure: ${df_expenditure['Total_Expense_2015_2020'].mean():,.0f}")
else:
    print("⚠ Year columns not found, checking column structure...")
    
print(f"\nExpenditure dataset shape: {df_expenditure.shape}")
df_expenditure.head()

In [ ]:
# Clean Physicians dataset
print("\nCleaning Physicians dataset...")

# Standardize state names
if 'State' in df_physicians.columns:
    df_physicians['State'] = df_physicians['State'].str.strip()
    print(f"✓ Standardized state names")
    
# Check for duplicates
duplicates = df_physicians.duplicated(subset=['State']).sum()
print(f"Duplicate states: {duplicates}")

if duplicates > 0:
    df_physicians = df_physicians.drop_duplicates(subset=['State'], keep='first')
    print(f"✓ Removed {duplicates} duplicate states")

print(f"Physicians dataset shape: {df_physicians.shape}")
print(f"Unique states: {df_physicians['State'].nunique()}")
df_physicians.head()

In [ ]:
# Clean Hospitals dataset
print("\nCleaning Hospitals dataset...")

# Standardize state names
if 'State' in df_hospitals.columns:
    df_hospitals['State'] = df_hospitals['State'].str.strip()
    print(f"✓ Standardized state names")
    
# Check for duplicates
duplicates_hosp = df_hospitals.duplicated(subset=['State']).sum()
print(f"Duplicate states: {duplicates_hosp}")

if duplicates_hosp > 0:
    df_hospitals = df_hospitals.drop_duplicates(subset=['State'], keep='first')
    print(f"✓ Removed {duplicates_hosp} duplicate states")

print(f"Hospitals dataset shape: {df_hospitals.shape}")
print(f"Unique states: {df_hospitals['State'].nunique()}")
df_hospitals.head()

In [ ]:
# Clean Population dataset
print("\nCleaning Population dataset...")

# Standardize state names
if 'State' in df_population.columns:
    df_population['State'] = df_population['State'].str.strip()
elif 'NAME' in df_population.columns:
    df_population = df_population.rename(columns={'NAME': 'State'})
    df_population['State'] = df_population['State'].str.strip()
    
print(f"✓ Standardized state names")
print(f"Population dataset shape: {df_population.shape}")
print(f"Unique states: {df_population['State'].nunique()}")
df_population.head()

In [ ]:
# Clean Physical Inactivity dataset  
print("\nCleaning Physical Inactivity dataset...")

# Select relevant columns
if 'LocationDesc' in df_inactivity.columns:
    df_inactivity = df_inactivity.rename(columns={'LocationDesc': 'State'})
    
df_inactivity['State'] = df_inactivity['State'].str.strip()

# Keep only overall stratification and remove US/territories
df_inactivity_clean = df_inactivity[
    (df_inactivity['State'] != 'United States') &
    (df_inactivity['State'] != 'Puerto Rico')
].copy()

print(f"✓ Cleaned Physical Inactivity dataset")
print(f"Shape: {df_inactivity_clean.shape}")
df_inactivity_clean.head()

In [ ]:
# Clean PM2.5 Pollution dataset
print("\nCleaning PM2.5 Pollution dataset...")

# Standardize state names
if 'State' in df_pollution.columns:
    df_pollution['State'] = df_pollution['State'].str.strip()
elif 'LocationDesc' in df_pollution.columns:
    df_pollution = df_pollution.rename(columns={'LocationDesc': 'State'})
    df_pollution['State'] = df_pollution['State'].str.strip()

# Remove US/territories
df_pollution_clean = df_pollution[
    (df_pollution['State'] != 'United States') &
    (df_pollution['State'] != 'Puerto Rico')
].copy()

print(f"✓ Cleaned PM2.5 Pollution dataset")
print(f"Shape: {df_pollution_clean.shape}")
df_pollution_clean.head()

In [ ]:
# Clean Per Capita Expenditure dataset
print("\nCleaning Per Capita Expenditure dataset...")

# Standardize state names
if 'State' in df_percapita.columns:
    df_percapita['State'] = df_percapita['State'].str.strip()
    
# Calculate total per capita spending 2015-2020 if year columns exist
year_cols_pc = ['Y2015', 'Y2016', 'Y2017', 'Y2018', 'Y2019', 'Y2020']
if all(col in df_percapita.columns for col in year_cols_pc):
    df_percapita['Total_PerCapita_2015_2020'] = df_percapita[year_cols_pc].sum(axis=1)
    print(f"✓ Calculated Total_PerCapita_2015_2020")
    
print(f"Per Capita dataset shape: {df_percapita.shape}")
df_percapita.head()

**✓ Commit 7 Complete:** All 7 datasets cleaned and standardized by Mithil.

## 10. Feature Engineering - Disease Summary Metrics

**Lead: Bharath Kumar A**

Creating aggregate disease metrics per state: Mean Disease Rate, Top Disease Topic, and Top Affected Gender.

In [ ]:
# Step 1: Calculate Mean Disease Rate per state
print("Calculating Mean Disease Rate per state...")

df_disease_summary = df_chronic_clean.groupby('State')['DataValue'].mean().reset_index()
df_disease_summary = df_disease_summary.rename(columns={'DataValue': 'MeanDiseaseRate'})

print(f"✓ Created disease summary for {df_disease_summary.shape[0]} states")
print(f"Mean Disease Rate range: {df_disease_summary['MeanDiseaseRate'].min():.2f} - {df_disease_summary['MeanDiseaseRate'].max():.2f}")
df_disease_summary.head()

In [ ]:
# Step 2: Identify Top Disease Topic per state
print("\nIdentifying Top Disease Topic per state...")

# Group by State and Topic, calculate mean disease value
topic_by_state = df_chronic_clean.groupby(['State', 'Topic'])['DataValue'].mean().reset_index()

# Find topic with highest mean value per state
top_topics = topic_by_state.loc[topic_by_state.groupby('State')['DataValue'].idxmax()]
top_topics = top_topics[['State', 'Topic']].rename(columns={'Topic': 'TopDiseaseTopic'})

# Merge with disease summary
df_disease_summary = df_disease_summary.merge(top_topics, on='State', how='left')

print(f"✓ Identified top disease topics")
print(f"\nMost common top disease topics:")
print(df_disease_summary['TopDiseaseTopic'].value_counts().head())
df_disease_summary.head()

In [ ]:
# Step 3: Identify Top Affected Gender per state
print("\nIdentifying Top Affected Gender per state...")

# Filter for gender stratification only
df_gender = df_chronic_clean[
    df_chronic_clean['StratificationCategory1'] == 'Sex'
].copy()

# Group by State and Gender
gender_by_state = df_gender.groupby(['State', 'Stratification1'])['DataValue'].mean().reset_index()

# Find gender with highest mean value per state
top_genders = gender_by_state.loc[gender_by_state.groupby('State')['DataValue'].idxmax()]
top_genders = top_genders[['State', 'Stratification1']].rename(columns={'Stratification1': 'TopAffectedGender'})

# Merge with disease summary
df_disease_summary = df_disease_summary.merge(top_genders, on='State', how='left')

print(f"✓ Identified top affected genders")
print(f"\nGender distribution:")
print(df_disease_summary['TopAffectedGender'].value_counts())

print(f"\n✓ Disease summary dataframe complete: {df_disease_summary.shape}")
df_disease_summary.head(10)

**✓ Commit 8 Complete:** Disease summary features engineered by Bharath.

## 11. Master Dataframe Integration

**Lead: Mithil Sri Sai Devisetty**

Merging all datasets into a single master dataframe for comprehensive state-level analysis.

In [ ]:
# Start with disease summary as base
df_master = df_disease_summary.copy()
print(f"Starting with disease summary: {df_master.shape}")

# Merge healthcare expenditure
if 'Total_Expense_2015_2020' in df_expenditure.columns:
    df_expense_merge = df_expenditure[['State', 'Total_Expense_2015_2020']].copy()
    df_master = df_master.merge(df_expense_merge, on='State', how='left')
    print(f"✓ Merged healthcare expenditure: {df_master.shape}")
else:
    print("⚠ Expenditure total column not found")

df_master.head()

In [ ]:
# Merge per capita expenditure
if 'Total_PerCapita_2015_2020' in df_percapita.columns:
    df_percap_merge = df_percapita[['State', 'Total_PerCapita_2015_2020']].copy()
    df_master = df_master.merge(df_percap_merge, on='State', how='left')
    print(f"✓ Merged per capita expenditure: {df_master.shape}")
else:
    print("⚠ Per capita total column not found")

df_master.head()

In [ ]:
# Merge physicians data
# Need to identify total physicians column
phys_cols = df_physicians.columns.tolist()
print(f"Physicians columns: {phys_cols}")

# Common patterns: 'Total', 'TotalPhysicians', 'ActivePhysicians'
total_phys_col = None
for col in phys_cols:
    if 'total' in col.lower() and 'physician' in col.lower():
        total_phys_col = col
        break
    elif col in ['TotalPhysicians', 'Total', 'ActivePhysicians']:
        total_phys_col = col
        break

if total_phys_col and total_phys_col in df_physicians.columns:
    df_phys_merge = df_physicians[['State', total_phys_col]].copy()
    df_phys_merge = df_phys_merge.rename(columns={total_phys_col: 'TotalPhysicians'})
    df_master = df_master.merge(df_phys_merge, on='State', how='left')
    print(f"✓ Merged physicians data using column '{total_phys_col}': {df_master.shape}")
else:
    print(f"⚠ Physicians total column not found, will calculate later")

df_master.head()

In [ ]:
# Merge hospitals data
hosp_cols = df_hospitals.columns.tolist()
print(f"Hospitals columns: {hosp_cols}")

# Common patterns: 'Total', 'TotalHospitals', 'Count'
total_hosp_col = None
for col in hosp_cols:
    if 'total' in col.lower() and 'hospital' in col.lower():
        total_hosp_col = col
        break
    elif col in ['TotalHospitals', 'Total', 'Count']:
        total_hosp_col = col
        break

if total_hosp_col and total_hosp_col in df_hospitals.columns:
    df_hosp_merge = df_hospitals[['State', total_hosp_col]].copy()
    df_hosp_merge = df_hosp_merge.rename(columns={total_hosp_col: 'TotalHospitals'})
    df_master = df_master.merge(df_hosp_merge, on='State', how='left')
    print(f"✓ Merged hospitals data using column '{total_hosp_col}': {df_master.shape}")
else:
    print(f"⚠ Hospitals total column not found, will calculate later")

df_master.head()

In [ ]:
# Merge population data
pop_cols = df_population.columns.tolist()
print(f"Population columns: {pop_cols}")

# Common patterns: 'Population', 'POPESTIMATE', 'POPESTIMATE2020'
pop_col = None
for col in pop_cols:
    if 'popestimate2020' in col.lower() or col == 'POPESTIMATE2020':
        pop_col = col
        break
    elif 'population' in col.lower():
        pop_col = col
        break
    elif 'POPESTIMATE' in col:
        pop_col = col
        break

if pop_col and pop_col in df_population.columns:
    df_pop_merge = df_population[['State', pop_col]].copy()
    df_pop_merge = df_pop_merge.rename(columns={pop_col: 'Population'})
    df_master = df_master.merge(df_pop_merge, on='State', how='left')
    print(f"✓ Merged population data using column '{pop_col}': {df_master.shape}")
else:
    print(f"⚠ Population column not found")

df_master.head()

In [ ]:
# Merge physical inactivity data
# Get overall stratification and aggregate by state
if 'Data_Value' in df_inactivity_clean.columns:
    inactivity_col = 'Data_Value'
elif 'DataValue' in df_inactivity_clean.columns:
    inactivity_col = 'DataValue'
else:
    inactivity_col = None

if inactivity_col:
    # Filter for overall stratification if exists
    if 'StratificationCategory1' in df_inactivity_clean.columns:
        df_inact_overall = df_inactivity_clean[
            df_inactivity_clean['StratificationCategory1'] == 'Overall'
        ].copy()
    else:
        df_inact_overall = df_inactivity_clean.copy()
    
    df_inact_merge = df_inact_overall.groupby('State')[inactivity_col].mean().reset_index()
    df_inact_merge = df_inact_merge.rename(columns={inactivity_col: 'Prevalence'})
    df_master = df_master.merge(df_inact_merge, on='State', how='left')
    print(f"✓ Merged physical inactivity data: {df_master.shape}")
else:
    print("⚠ Physical inactivity value column not found")

df_master.head()

In [ ]:
# Merge PM2.5 pollution data
if 'Data_Value' in df_pollution_clean.columns:
    pollution_col = 'Data_Value'
elif 'DataValue' in df_pollution_clean.columns:
    pollution_col = 'DataValue'
elif 'Value' in df_pollution_clean.columns:
    pollution_col = 'Value'
else:
    pollution_col = None

if pollution_col:
    df_poll_merge = df_pollution_clean.groupby('State')[pollution_col].mean().reset_index()
    df_poll_merge = df_poll_merge.rename(columns={pollution_col: 'PM25'})
    df_master = df_master.merge(df_poll_merge, on='State', how='left')
    print(f"✓ Merged PM2.5 pollution data: {df_master.shape}")
else:
    print("⚠ Pollution value column not found")

print(f"\n✓✓ Master dataframe complete: {df_master.shape}")
print(f"Columns: {df_master.columns.tolist()}")
df_master.head()

In [ ]:
# Check for missing values in master dataframe
print("\nMissing values in master dataframe:")
print(df_master.isnull().sum())

# Display summary statistics
print("\n✓ Master Dataframe Summary:")
df_master.describe()

**✓ Commit 9 Complete:** Master dataframe integrated by Mithil.

## 12. Advanced Feature Engineering - Per-Capita and Per-100k Metrics

**Lead: Tej Kiran Yenugunti**

Calculating normalized metrics to enable fair cross-state comparisons.

In [ ]:
# Calculate Physicians per 100k population
if 'TotalPhysicians' in df_master.columns and 'Population' in df_master.columns:
    df_master['Physicians_Per100k'] = (df_master['TotalPhysicians'] / df_master['Population']) * 100000
    print(f"✓ Calculated Physicians_Per100k")
    print(f"Range: {df_master['Physicians_Per100k'].min():.2f} - {df_master['Physicians_Per100k'].max():.2f}")
else:
    print("⚠ Cannot calculate Physicians_Per100k - missing required columns")

df_master.head()

In [ ]:
# Calculate Hospitals per 100k population
if 'TotalHospitals' in df_master.columns and 'Population' in df_master.columns:
    df_master['Hospitals_Per100k'] = (df_master['TotalHospitals'] / df_master['Population']) * 100000
    print(f"✓ Calculated Hospitals_Per100k")
    print(f"Range: {df_master['Hospitals_Per100k'].min():.2f} - {df_master['Hospitals_Per100k'].max():.2f}")
else:
    print("⚠ Cannot calculate Hospitals_Per100k - missing required columns")

df_master.head()

In [ ]:
# If per capita spending already exists, use it; otherwise calculate from total
if 'Total_PerCapita_2015_2020' not in df_master.columns:
    if 'Total_Expense_2015_2020' in df_master.columns and 'Population' in df_master.columns:
        df_master['Total_PerCapita_2015_2020'] = df_master['Total_Expense_2015_2020'] / df_master['Population']
        print(f"✓ Calculated Total_PerCapita_2015_2020 from total expense")
    else:
        print("⚠ Cannot calculate per capita spending - missing required columns")
else:
    print(f"✓ Per capita spending already exists")

if 'Total_PerCapita_2015_2020' in df_master.columns:
    print(f"Range: ${df_master['Total_PerCapita_2015_2020'].min():.2f} - ${df_master['Total_PerCapita_2015_2020'].max():.2f}")

df_master.head()

In [ ]:
# Handle any remaining missing values with median imputation for numeric columns
print("\nHandling missing values...")

numeric_cols = df_master.select_dtypes(include=[np.number]).columns.tolist()
missing_before = df_master[numeric_cols].isnull().sum().sum()

for col in numeric_cols:
    if df_master[col].isnull().sum() > 0:
        median_value = df_master[col].median()
        df_master[col].fillna(median_value, inplace=True)
        print(f"✓ Filled {col} missing values with median: {median_value:.2f}")

missing_after = df_master[numeric_cols].isnull().sum().sum()
print(f"\n✓ Missing values reduced from {missing_before} to {missing_after}")

print(f"\n✓✓ Final master dataframe: {df_master.shape}")
print(f"Columns: {len(df_master.columns)}")
df_master.head()

**✓ Commit 10 Complete:** Per-capita and per-100k metrics calculated by Tej.

## 13. Exploratory Data Analysis (EDA)

**Lead: Tej Kiran Yenugunti**

Comprehensive visualization and statistical analysis of key variables and relationships.

In [ ]:
# Distribution of Mean Disease Rate across states
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(df_master['MeanDiseaseRate'], bins=20, edgecolor='black', alpha=0.7)
ax.axvline(df_master['MeanDiseaseRate'].mean(), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {df_master["MeanDiseaseRate"].mean():.2f}')
ax.axvline(df_master['MeanDiseaseRate'].median(), color='green', linestyle='--', 
           linewidth=2, label=f'Median: {df_master["MeanDiseaseRate"].median():.2f}')

ax.set_xlabel('Mean Disease Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of States', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Mean Disease Rate Across U.S. States', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/eda_disease_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Mean Disease Rate - Mean: {df_master['MeanDiseaseRate'].mean():.2f}, Median: {df_master['MeanDiseaseRate'].median():.2f}, Std: {df_master['MeanDiseaseRate'].std():.2f}")

In [ ]:
# Top Disease Topics Distribution
fig, ax = plt.subplots(figsize=(14, 6))

topic_counts = df_master['TopDiseaseTopic'].value_counts()
topic_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')

ax.set_xlabel('Disease Topic', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of States', fontsize=12, fontweight='bold')
ax.set_title('Top Disease Topic Distribution Across U.S. States', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/eda_top_topics.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Most common top disease topic: {topic_counts.index[0]} ({topic_counts.values[0]} states)")

In [ ]:
# Correlation Heatmap of Key Numeric Variables
numeric_features = df_master.select_dtypes(include=[np.number]).columns.tolist()

# Remove categorical IDs if any
features_for_corr = [f for f in numeric_features if f != 'State']

fig, ax = plt.subplots(figsize=(12, 10))

correlation_matrix = df_master[features_for_corr].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, ax=ax, 
            cbar_kws={"shrink": 0.8})

ax.set_title('Correlation Matrix - Chronic Disease & Health Factors', 
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../results/eda_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Correlation heatmap generated")

In [ ]:
# Scatter plot: Per Capita Spending vs Disease Rate
if 'Total_PerCapita_2015_2020' in df_master.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    scatter = ax.scatter(df_master['Total_PerCapita_2015_2020'], 
                         df_master['MeanDiseaseRate'],
                         alpha=0.6, s=100, c=df_master['MeanDiseaseRate'],
                         cmap='YlOrRd', edgecolor='black')
    
    # Add trendline
    z = np.polyfit(df_master['Total_PerCapita_2015_2020'], df_master['MeanDiseaseRate'], 1)
    p = np.poly1d(z)
    ax.plot(df_master['Total_PerCapita_2015_2020'], 
            p(df_master['Total_PerCapita_2015_2020']), 
            "r--", linewidth=2, label='Trend Line')
    
    ax.set_xlabel('Total Per Capita Spending (2015-2020)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Disease Rate', fontsize=12, fontweight='bold')
    ax.set_title('Healthcare Spending vs Disease Burden', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(scatter, label='Disease Rate', ax=ax)
    plt.tight_layout()
    plt.savefig('../results/eda_spending_vs_disease.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    correlation = df_master['Total_PerCapita_2015_2020'].corr(df_master['MeanDiseaseRate'])
    print(f"Correlation (Spending vs Disease): {correlation:.3f}")
else:
    print("⚠ Per capita spending column not available")

In [ ]:
# Box plots for key variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Mean Disease Rate
axes[0, 0].boxplot(df_master['MeanDiseaseRate'], vert=True)
axes[0, 0].set_ylabel('Mean Disease Rate', fontweight='bold')
axes[0, 0].set_title('Disease Rate Distribution', fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: Physicians per 100k
if 'Physicians_Per100k' in df_master.columns:
    axes[0, 1].boxplot(df_master['Physicians_Per100k'].dropna(), vert=True)
    axes[0, 1].set_ylabel('Physicians per 100k', fontweight='bold')
    axes[0, 1].set_title('Physician Density Distribution', fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Physical Inactivity Prevalence
if 'Prevalence' in df_master.columns:
    axes[1, 0].boxplot(df_master['Prevalence'].dropna(), vert=True)
    axes[1, 0].set_ylabel('Inactivity Prevalence (%)', fontweight='bold')
    axes[1, 0].set_title('Physical Inactivity Distribution', fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 4: PM2.5 Pollution
if 'PM25' in df_master.columns:
    axes[1, 1].boxplot(df_master['PM25'].dropna(), vert=True)
    axes[1, 1].set_ylabel('PM2.5 Level', fontweight='bold')
    axes[1, 1].set_title('Air Pollution Distribution', fontweight='bold')
    axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/eda_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Box plots generated for key variables")

**✓ Commit 11 Complete:** Core EDA visualizations created by Tej.

## 14. Research Questions - Q1: Healthcare Spending Analysis

**Lead: Mithil Sri Sai Devisetty**

Investigating the relationship between healthcare spending and disease outcomes.

In [ ]:
### Q1.1: Does higher per-capita spending reduce disease burden?

if 'Total_PerCapita_2015_2020' in df_master.columns:
    print("Q1.1: Per-Capita Spending vs Disease Burden")
    
    correlation_percap = df_master['Total_PerCapita_2015_2020'].corr(df_master['MeanDiseaseRate'])
    print(f"Correlation: {correlation_percap:.3f}")
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    scatter = ax.scatter(df_master['Total_PerCapita_2015_2020'], 
                         df_master['MeanDiseaseRate'],
                         s=120, alpha=0.6, c=df_master['MeanDiseaseRate'],
                         cmap='RdYlGn_r', edgecolor='black', linewidth=0.5)
    
    # Add trendline
    z = np.polyfit(df_master['Total_PerCapita_2015_2020'], df_master['MeanDiseaseRate'], 1)
    p = np.poly1d(z)
    ax.plot(df_master['Total_PerCapita_2015_2020'], 
            p(df_master['Total_PerCapita_2015_2020']), 
            "r--", linewidth=2, alpha=0.8, label='Trend Line')
    
    ax.set_xlabel('Total Per Capita Spending (2015-2020)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Disease Rate', fontsize=12, fontweight='bold')
    ax.set_title(f'Q1.1: Per-Capita Spending vs Disease Burden\n(Correlation: {correlation_percap:.3f})',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(scatter, label='Disease Rate', ax=ax)
    plt.tight_layout()
    plt.savefig('../results/q1_1_spending_vs_disease.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠ Per capita spending data not available")

In [ ]:
### Q1.2: Is there a point of diminishing returns on healthcare spending?

if 'Total_PerCapita_2015_2020' in df_master.columns:
    print("\nQ1.2: Diminishing Returns on Healthcare Spending")
    
    # Fit polynomial curve (degree 2 for diminishing returns)
    X_spending = df_master['Total_PerCapita_2015_2020'].values.reshape(-1, 1)
    y_disease = df_master['MeanDiseaseRate'].values
    
    # Polynomial regression (degree 2)
    from sklearn.preprocessing import PolynomialFeatures
    poly = PolynomialFeatures(degree=2)
    X_poly = poly.fit_transform(X_spending)
    
    from sklearn.linear_model import LinearRegression
    poly_model = LinearRegression()
    poly_model.fit(X_poly, y_disease)
    
    # Generate smooth curve for plotting
    X_range = np.linspace(X_spending.min(), X_spending.max(), 300).reshape(-1, 1)
    X_range_poly = poly.transform(X_range)
    y_pred = poly_model.predict(X_range_poly)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    ax.scatter(df_master['Total_PerCapita_2015_2020'], df_master['MeanDiseaseRate'],
               alpha=0.6, s=100, edgecolor='black', label='States')
    ax.plot(X_range, y_pred, 'r-', linewidth=3, label='Polynomial Fit (Degree 2)')
    
    ax.set_xlabel('Total Per Capita Spending (2015-2020)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Disease Rate', fontsize=12, fontweight='bold')
    ax.set_title('Q1.2: Diminishing Returns on Healthcare Spending', fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/q1_2_diminishing_returns.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Polynomial regression analysis complete")

In [ ]:
### Q1.3: Which states achieve the best health outcomes per dollar spent? (Efficiency Analysis)

if 'Total_PerCapita_2015_2020' in df_master.columns:
    print("\nQ1.3: Healthcare Spending Efficiency Analysis")
    
    # Calculate median values for quadrants
    median_spending = df_master['Total_PerCapita_2015_2020'].median()
    median_disease = df_master['MeanDiseaseRate'].median()
    
    # Classify states into quadrants
    df_master['Efficiency_Quadrant'] = 'Unknown'
    
    # Low spending, low disease = HIGH EFFICIENCY
    df_master.loc[(df_master['Total_PerCapita_2015_2020'] < median_spending) & 
                  (df_master['MeanDiseaseRate'] < median_disease), 'Efficiency_Quadrant'] = 'High Efficiency'
    
    # High spending, low disease = High spending, good outcomes
    df_master.loc[(df_master['Total_PerCapita_2015_2020'] >= median_spending) & 
                  (df_master['MeanDiseaseRate'] < median_disease), 'Efficiency_Quadrant'] = 'High Spending, Good Outcomes'
    
    # Low spending, high disease = Low spending, poor outcomes
    df_master.loc[(df_master['Total_PerCapita_2015_2020'] < median_spending) & 
                  (df_master['MeanDiseaseRate'] >= median_disease), 'Efficiency_Quadrant'] = 'Low Spending, Poor Outcomes'
    
    # High spending, high disease = LOW EFFICIENCY
    df_master.loc[(df_master['Total_PerCapita_2015_2020'] >= median_spending) & 
                  (df_master['MeanDiseaseRate'] >= median_disease), 'Efficiency_Quadrant'] = 'Low Efficiency'
    
    # Plot efficiency quadrants
    fig, ax = plt.subplots(figsize=(14, 9))
    
    colors = {'High Efficiency': 'green', 
              'High Spending, Good Outcomes': 'blue',
              'Low Spending, Poor Outcomes': 'orange',
              'Low Efficiency': 'red'}
    
    for quadrant, color in colors.items():
        mask = df_master['Efficiency_Quadrant'] == quadrant
        ax.scatter(df_master[mask]['Total_PerCapita_2015_2020'],
                   df_master[mask]['MeanDiseaseRate'],
                   c=color, label=quadrant, s=150, alpha=0.7, edgecolor='black')
    
    # Add median lines
    ax.axvline(median_spending, color='gray', linestyle='--', linewidth=2, alpha=0.7, label='Median Spending')
    ax.axhline(median_disease, color='gray', linestyle='--', linewidth=2, alpha=0.7, label='Median Disease')
    
    ax.set_xlabel('Total Per Capita Spending (2015-2020)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Disease Rate', fontsize=12, fontweight='bold')
    ax.set_title('Q1.3: Healthcare Spending Efficiency Quadrant Analysis',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/q1_3_efficiency_quadrants.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nEfficiency Distribution:")
    print(df_master['Efficiency_Quadrant'].value_counts())
    
    print("\n✓ High Efficiency States (Low Spending, Low Disease):")
    high_eff = df_master[df_master['Efficiency_Quadrant'] == 'High Efficiency']['State'].tolist()
    print(f"   {', '.join(high_eff[:10])}")  # Show first 10

**✓ Commit 12 Complete:** Q1 healthcare spending analysis by Mithil.

## 15. Research Questions - Q2: Healthcare Workforce Analysis

**Lead: Mithil Sri Sai Devisetty**

Analyzing the impact of physician and hospital density on health outcomes.

In [ ]:
### Q2.1: Do more physicians per 100k residents lead to lower disease burden?

if 'Physicians_Per100k' in df_master.columns:
    print("Q2.1: Physicians per 100k vs Disease Burden")
    
    correlation_phys = df_master['Physicians_Per100k'].corr(df_master['MeanDiseaseRate'])
    print(f"Correlation: {correlation_phys:.3f}")
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    scatter = ax.scatter(df_master['Physicians_Per100k'], 
                         df_master['MeanDiseaseRate'],
                         s=120, alpha=0.6, c=df_master['MeanDiseaseRate'],
                         cmap='YlOrRd', edgecolor='black', linewidth=0.5)
    
    # Add trendline
    z = np.polyfit(df_master['Physicians_Per100k'].dropna(), 
                   df_master.loc[df_master['Physicians_Per100k'].notna(), 'MeanDiseaseRate'], 1)
    p = np.poly1d(z)
    ax.plot(df_master['Physicians_Per100k'].dropna(), 
            p(df_master['Physicians_Per100k'].dropna()), 
            "r--", linewidth=2, alpha=0.8, label='Trend Line')
    
    ax.set_xlabel('Physicians per 100k Population', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Disease Rate', fontsize=12, fontweight='bold')
    ax.set_title(f'Q2.1: Physicians per 100k vs Disease Burden\n(Correlation: {correlation_phys:.3f})',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(scatter, label='Disease Rate', ax=ax)
    plt.tight_layout()
    plt.savefig('../results/q2_1_physicians_vs_disease.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠ Physicians per 100k data not available")

In [ ]:
### Q2.2: Do hospital resources correlate with better outcomes?

if 'Hospitals_Per100k' in df_master.columns:
    print("\nQ2.2: Hospitals per 100k vs Disease Burden")
    
    correlation_hosp = df_master['Hospitals_Per100k'].corr(df_master['MeanDiseaseRate'])
    print(f"Correlation: {correlation_hosp:.3f}")
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    scatter = ax.scatter(df_master['Hospitals_Per100k'], 
                         df_master['MeanDiseaseRate'],
                         s=120, alpha=0.6, c=df_master['MeanDiseaseRate'],
                         cmap='RdYlGn_r', edgecolor='black', linewidth=0.5)
    
    # Add trendline
    z = np.polyfit(df_master['Hospitals_Per100k'].dropna(), 
                   df_master.loc[df_master['Hospitals_Per100k'].notna(), 'MeanDiseaseRate'], 1)
    p = np.poly1d(z)
    ax.plot(df_master['Hospitals_Per100k'].dropna(), 
            p(df_master['Hospitals_Per100k'].dropna()), 
            "r--", linewidth=2, alpha=0.8, label='Trend Line')
    
    ax.set_xlabel('Hospitals per 100k Population', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Disease Rate', fontsize=12, fontweight='bold')
    ax.set_title(f'Q2.2: Hospitals per 100k vs Disease Burden\n(Correlation: {correlation_hosp:.3f})',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(scatter, label='Disease Rate', ax=ax)
    plt.tight_layout()
    plt.savefig('../results/q2_2_hospitals_vs_disease.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠ Hospitals per 100k data not available")

In [ ]:
### Q2.3: Interaction Effects - Spending & Physician Density

if 'Total_PerCapita_2015_2020' in df_master.columns and 'Physicians_Per100k' in df_master.columns:
    print("\nQ2.3: Interaction Effects - Spending & Physician Density")
    
    # Create 3D plot
    from mpl_toolkits.mplot3d import Axes3D
    
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    scatter = ax.scatter(df_master['Total_PerCapita_2015_2020'],
                         df_master['Physicians_Per100k'],
                         df_master['MeanDiseaseRate'],
                         c=df_master['MeanDiseaseRate'], cmap='RdYlGn_r',
                         s=150, alpha=0.7, edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Per Capita Spending', fontsize=11, fontweight='bold')
    ax.set_ylabel('Physicians per 100k', fontsize=11, fontweight='bold')
    ax.set_zlabel('Mean Disease Rate', fontsize=11, fontweight='bold')
    ax.set_title('Q2.3: Interaction Effects - Spending & Physician Density\non Disease Burden',
                 fontsize=13, fontweight='bold', pad=20)
    
    fig.colorbar(scatter, label='Disease Rate', shrink=0.5, aspect=10)
    plt.tight_layout()
    plt.savefig('../results/q2_3_interaction_effects.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ 3D interaction analysis complete")
else:
    print("⚠ Required data not available for interaction analysis")

**✓ Commit 13 Complete:** Q2 workforce composition analysis by Mithil.

## 16. Research Questions - Q3: Environmental and Behavioral Factors

**Lead: Mani Mokshith Noonety**

Analyzing the impact of physical inactivity and air pollution on chronic disease burden.

In [ ]:
### Q3.1: Does physical inactivity predict chronic disease burden?

if 'Prevalence' in df_master.columns:
    print("Q3.1: Physical Inactivity vs Disease Burden")
    
    correlation_inactivity = df_master['Prevalence'].corr(df_master['MeanDiseaseRate'])
    print(f"Correlation: {correlation_inactivity:.3f}")
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    scatter = ax.scatter(df_master['Prevalence'], 
                         df_master['MeanDiseaseRate'],
                         s=120, alpha=0.6, c=df_master['MeanDiseaseRate'],
                         cmap='Reds', edgecolor='black', linewidth=0.5)
    
    # Add trendline
    mask = df_master['Prevalence'].notna() & df_master['MeanDiseaseRate'].notna()
    z = np.polyfit(df_master.loc[mask, 'Prevalence'], 
                   df_master.loc[mask, 'MeanDiseaseRate'], 1)
    p = np.poly1d(z)
    ax.plot(df_master.loc[mask, 'Prevalence'], 
            p(df_master.loc[mask, 'Prevalence']), 
            "r--", linewidth=2, alpha=0.8, label='Trend Line')
    
    ax.set_xlabel('Physical Inactivity Prevalence (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Disease Rate', fontsize=12, fontweight='bold')
    ax.set_title(f'Q3.1: Physical Inactivity vs Disease Burden\n(Correlation: {correlation_inactivity:.3f})',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(scatter, label='Disease Rate', ax=ax)
    plt.tight_layout()
    plt.savefig('../results/q3_1_inactivity_vs_disease.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠ Physical inactivity data not available")

In [ ]:
### Q3.2: Does air pollution (PM2.5) correlate with disease burden?

if 'PM25' in df_master.columns:
    print("\nQ3.2: Air Pollution (PM2.5) vs Disease Burden")
    
    correlation_pollution = df_master['PM25'].corr(df_master['MeanDiseaseRate'])
    print(f"Correlation: {correlation_pollution:.3f}")
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    scatter = ax.scatter(df_master['PM25'], 
                         df_master['MeanDiseaseRate'],
                         s=120, alpha=0.6, c=df_master['MeanDiseaseRate'],
                         cmap='YlOrBr', edgecolor='black', linewidth=0.5)
    
    # Add trendline
    mask = df_master['PM25'].notna() & df_master['MeanDiseaseRate'].notna()
    z = np.polyfit(df_master.loc[mask, 'PM25'], 
                   df_master.loc[mask, 'MeanDiseaseRate'], 1)
    p = np.poly1d(z)
    ax.plot(df_master.loc[mask, 'PM25'], 
            p(df_master.loc[mask, 'PM25']), 
            "r--", linewidth=2, alpha=0.8, label='Trend Line')
    
    ax.set_xlabel('PM2.5 Air Pollution Level', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Disease Rate', fontsize=12, fontweight='bold')
    ax.set_title(f'Q3.2: Air Pollution vs Disease Burden\n(Correlation: {correlation_pollution:.3f})',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(scatter, label='Disease Rate', ax=ax)
    plt.tight_layout()
    plt.savefig('../results/q3_2_pollution_vs_disease.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠ PM2.5 pollution data not available")

In [ ]:
### Q3.3: Do environmental factors (pollution + inactivity) compound each other's effects?

if 'Prevalence' in df_master.columns and 'PM25' in df_master.columns:
    print("\nQ3.3: Combined Environmental Impact")
    
    # Create 3D plot
    from mpl_toolkits.mplot3d import Axes3D
    
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    scatter = ax.scatter(df_master['Prevalence'],
                         df_master['PM25'],
                         df_master['MeanDiseaseRate'],
                         c=df_master['MeanDiseaseRate'], cmap='RdYlGn_r',
                         s=150, alpha=0.7, edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Physical Inactivity (%)', fontsize=11, fontweight='bold')
    ax.set_ylabel('PM2.5 Pollution', fontsize=11, fontweight='bold')
    ax.set_zlabel('Mean Disease Rate', fontsize=11, fontweight='bold')
    ax.set_title('Q3.3: Combined Environmental Factors\nInactivity + Pollution vs Disease Burden',
                 fontsize=13, fontweight='bold', pad=20)
    
    fig.colorbar(scatter, label='Disease Rate', shrink=0.5, aspect=10)
    plt.tight_layout()
    plt.savefig('../results/q3_3_combined_environmental.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Combined environmental analysis complete")
else:
    print("⚠ Environmental data not complete for analysis")

**✓ Commit 14 Complete:** Q3 environmental factors analysis by Mani.

## 17. Machine Learning - Predictive Modeling

**Lead: Bharath Kumar A**

Building supervised learning models to predict chronic disease burden based on multiple factors.

In [ ]:
# Prepare data for machine learning
print("Preparing data for ML models...")

# Select features for prediction
feature_cols = []
if 'Total_PerCapita_2015_2020' in df_master.columns:
    feature_cols.append('Total_PerCapita_2015_2020')
if 'Physicians_Per100k' in df_master.columns:
    feature_cols.append('Physicians_Per100k')
if 'Hospitals_Per100k' in df_master.columns:
    feature_cols.append('Hospitals_Per100k')
if 'Prevalence' in df_master.columns:
    feature_cols.append('Prevalence')
if 'PM25' in df_master.columns:
    feature_cols.append('PM25')

target_col = 'MeanDiseaseRate'

print(f"Features for prediction: {feature_cols}")
print(f"Target variable: {target_col}")

# Create feature matrix and target vector
X = df_master[feature_cols].copy()
y = df_master[target_col].copy()

# Handle any remaining missing values
X = X.fillna(X.median())

print(f"\n✓ Feature matrix shape: {X.shape}")
print(f"✓ Target vector shape: {y.shape}")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Data split and scaled successfully")

In [ ]:
### Model 1: Linear Regression

print("\n" + "="*60)
print("MODEL 1: LINEAR REGRESSION")
print("="*60)

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr_test = lr_model.predict(X_test_scaled)

# Evaluate model
r2_train_lr = r2_score(y_train, y_pred_lr_train)
r2_test_lr = r2_score(y_test, y_pred_lr_test)
rmse_train_lr = np.sqrt(mean_squared_error(y_train, y_pred_lr_train))
rmse_test_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr_test))
mae_train_lr = mean_absolute_error(y_train, y_pred_lr_train)
mae_test_lr = mean_absolute_error(y_test, y_pred_lr_test)

print(f"\nTraining Performance:")
print(f"  R² Score: {r2_train_lr:.4f}")
print(f"  RMSE: {rmse_train_lr:.4f}")
print(f"  MAE: {mae_train_lr:.4f}")

print(f"\nTesting Performance:")
print(f"  R² Score: {r2_test_lr:.4f}")
print(f"  RMSE: {rmse_test_lr:.4f}")
print(f"  MAE: {mae_test_lr:.4f}")

# Feature importance (coefficients)
print(f"\nFeature Coefficients:")
for feature, coef in zip(feature_cols, lr_model.coef_):
    print(f"  {feature}: {coef:.4f}")

In [ ]:
### Model 2: Random Forest Regressor

print("\n" + "="*60)
print("MODEL 2: RANDOM FOREST")
print("="*60)

# Train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_rf_train = rf_model.predict(X_train_scaled)
y_pred_rf_test = rf_model.predict(X_test_scaled)

# Evaluate model
r2_train_rf = r2_score(y_train, y_pred_rf_train)
r2_test_rf = r2_score(y_test, y_pred_rf_test)
rmse_train_rf = np.sqrt(mean_squared_error(y_train, y_pred_rf_train))
rmse_test_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf_test))
mae_train_rf = mean_absolute_error(y_train, y_pred_rf_train)
mae_test_rf = mean_absolute_error(y_test, y_pred_rf_test)

print(f"\nTraining Performance:")
print(f"  R² Score: {r2_train_rf:.4f}")
print(f"  RMSE: {rmse_train_rf:.4f}")
print(f"  MAE: {mae_train_rf:.4f}")

print(f"\nTesting Performance:")
print(f"  R² Score: {r2_test_rf:.4f}")
print(f"  RMSE: {rmse_test_rf:.4f}")
print(f"  MAE: {mae_test_rf:.4f}")

# Feature importance
print(f"\nFeature Importances:")
for feature, importance in zip(feature_cols, rf_model.feature_importances_):
    print(f"  {feature}: {importance:.4f}")

**✓ Commit 15 Complete:** Linear Regression and Random Forest models implemented.

In [ ]:
### Model 3: Gradient Boosting Regressor

print("\n" + "="*60)
print("MODEL 3: GRADIENT BOOSTING")
print("="*60)

# Train Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1)
gb_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_gb_train = gb_model.predict(X_train_scaled)
y_pred_gb_test = gb_model.predict(X_test_scaled)

# Evaluate model
r2_train_gb = r2_score(y_train, y_pred_gb_train)
r2_test_gb = r2_score(y_test, y_pred_gb_test)
rmse_train_gb = np.sqrt(mean_squared_error(y_train, y_pred_gb_train))
rmse_test_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb_test))
mae_train_gb = mean_absolute_error(y_train, y_pred_gb_train)
mae_test_gb = mean_absolute_error(y_test, y_pred_gb_test)

print(f"\nTraining Performance:")
print(f"  R² Score: {r2_train_gb:.4f}")
print(f"  RMSE: {rmse_train_gb:.4f}")
print(f"  MAE: {mae_train_gb:.4f}")

print(f"\nTesting Performance:")
print(f"  R² Score: {r2_test_gb:.4f}")
print(f"  RMSE: {rmse_test_gb:.4f}")
print(f"  MAE: {mae_test_gb:.4f}")

# Feature importance
print(f"\nFeature Importances:")
for feature, importance in zip(feature_cols, gb_model.feature_importances_):
    print(f"  {feature}: {importance:.4f}")

In [ ]:
### Model 4: XGBoost Regressor

print("\n" + "="*60)
print("MODEL 4: XGBOOST")
print("="*60)

# Train XGBoost
xgb_model = XGBRegressor(n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1)
xgb_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_xgb_train = xgb_model.predict(X_train_scaled)
y_pred_xgb_test = xgb_model.predict(X_test_scaled)

# Evaluate model
r2_train_xgb = r2_score(y_train, y_pred_xgb_train)
r2_test_xgb = r2_score(y_test, y_pred_xgb_test)
rmse_train_xgb = np.sqrt(mean_squared_error(y_train, y_pred_xgb_train))
rmse_test_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb_test))
mae_train_xgb = mean_absolute_error(y_train, y_pred_xgb_train)
mae_test_xgb = mean_absolute_error(y_test, y_pred_xgb_test)

print(f"\nTraining Performance:")
print(f"  R² Score: {r2_train_xgb:.4f}")
print(f"  RMSE: {rmse_train_xgb:.4f}")
print(f"  MAE: {mae_train_xgb:.4f}")

print(f"\nTesting Performance:")
print(f"  R² Score: {r2_test_xgb:.4f}")
print(f"  RMSE: {rmse_test_xgb:.4f}")
print(f"  MAE: {mae_test_xgb:.4f}")

# Feature importance
print(f"\nFeature Importances:")
for feature, importance in zip(feature_cols, xgb_model.feature_importances_):
    print(f"  {feature}: {importance:.4f}")

In [ ]:
# Model Comparison Summary

print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

models_summary = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost'],
    'Train R²': [r2_train_lr, r2_train_rf, r2_train_gb, r2_train_xgb],
    'Test R²': [r2_test_lr, r2_test_rf, r2_test_gb, r2_test_xgb],
    'Train RMSE': [rmse_train_lr, rmse_train_rf, rmse_train_gb, rmse_train_xgb],
    'Test RMSE': [rmse_test_lr, rmse_test_rf, rmse_test_gb, rmse_test_xgb],
    'Test MAE': [mae_test_lr, mae_test_rf, mae_test_gb, mae_test_xgb]
})

print("\n", models_summary.to_string(index=False))

# Find best model
best_model_idx = models_summary['Test R²'].idxmax()
best_model_name = models_summary.loc[best_model_idx, 'Model']
print(f"\n✓ Best Model: {best_model_name} (Test R² = {models_summary.loc[best_model_idx, 'Test R²']:.4f})")

In [ ]:
# Visualization: Model Comparison

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

models_data = [
    ('Linear Regression', y_pred_lr_test, r2_test_lr, rmse_test_lr),
    ('Random Forest', y_pred_rf_test, r2_test_rf, rmse_test_rf),
    ('Gradient Boosting', y_pred_gb_test, r2_test_gb, rmse_test_gb),
    ('XGBoost', y_pred_xgb_test, r2_test_xgb, rmse_test_xgb)
]

for idx, (name, y_pred, r2, rmse) in enumerate(models_data):
    row = idx // 2
    col = idx % 2
    
    axes[row, col].scatter(y_test, y_pred, alpha=0.6, s=100, edgecolor='black')
    axes[row, col].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
                        'r--', lw=2, label='Perfect Prediction')
    axes[row, col].set_xlabel('Actual Disease Rate', fontsize=11, fontweight='bold')
    axes[row, col].set_ylabel('Predicted Disease Rate', fontsize=11, fontweight='bold')
    axes[row, col].set_title(f'{name}\nR² = {r2:.4f}, RMSE = {rmse:.4f}',
                             fontsize=12, fontweight='bold')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/ml_all_models_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ All model visualizations complete")

**✓ Commit 16 Complete:** XGBoost and Gradient Boosting models implemented.

## 18. Unsupervised Learning - K-Means Clustering

Identifying state archetypes based on health and resource profiles.

In [ ]:
# Prepare data for clustering
print("Preparing data for clustering...")

# Use the same features as ML models
X_clustering = df_master[feature_cols].copy()
X_clustering = X_clustering.fillna(X_clustering.median())

# Standardize features
scaler_cluster = StandardScaler()
X_clustering_scaled = scaler_cluster.fit_transform(X_clustering)

print(f"✓ Clustering data prepared: {X_clustering_scaled.shape}")

In [ ]:
# Elbow Method to find optimal K
print("\nPerforming Elbow Method analysis...")

inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_clustering_scaled)
    inertias.append(kmeans.inertia_)

# Plot Elbow Curve
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (K)', fontsize=12, fontweight='bold')
ax.set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12, fontweight='bold')
ax.set_title('Elbow Method For Optimal K', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/clustering_elbow_method.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Elbow method complete")

In [ ]:
# Silhouette Analysis
print("\nPerforming Silhouette Analysis...")

from sklearn.metrics import silhouette_score

silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_clustering_scaled)
    silhouette_avg = silhouette_score(X_clustering_scaled, cluster_labels)
    silhouette_scores.append(silhouette_avg)
    print(f"  K={k}: Silhouette Score = {silhouette_avg:.4f}")

# Plot Silhouette Scores
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (K)', fontsize=12, fontweight='bold')
ax.set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
ax.set_title('Silhouette Analysis For Optimal K', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/clustering_silhouette_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Find optimal K
optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\n✓ Optimal K based on Silhouette Score: {optimal_k}")

In [ ]:
# Perform K-Means with optimal K
print(f"\nPerforming K-Means clustering with K={optimal_k}...")

kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_master['Cluster'] = kmeans_final.fit_predict(X_clustering_scaled)

print(f"✓ K-Means clustering complete")
print(f"\nCluster Distribution:")
print(df_master['Cluster'].value_counts().sort_index())

# Cluster centers
cluster_centers = scaler_cluster.inverse_transform(kmeans_final.cluster_centers_)
cluster_centers_df = pd.DataFrame(cluster_centers, columns=feature_cols)
cluster_centers_df['Cluster'] = range(optimal_k)

print(f"\n✓ Cluster Centers:")
print(cluster_centers_df.to_string(index=False))

In [ ]:
# PCA Visualization of Clusters
print("\nPerforming PCA for 2D visualization...")

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_clustering_scaled)

# Add PCA components to dataframe
df_master['PCA1'] = X_pca[:, 0]
df_master['PCA2'] = X_pca[:, 1]

print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.3f}")

# Plot PCA clusters
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(df_master['PCA1'], df_master['PCA2'], 
                     c=df_master['Cluster'], cmap='viridis',
                     s=150, alpha=0.7, edgecolor='black', linewidth=0.5)

# Add cluster centers
pca_centers = pca.transform(kmeans_final.cluster_centers_)
ax.scatter(pca_centers[:, 0], pca_centers[:, 1], 
           c='red', marker='X', s=500, edgecolor='black', linewidth=2,
           label='Cluster Centers')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', 
              fontsize=12, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', 
              fontsize=12, fontweight='bold')
ax.set_title('K-Means Clustering - PCA Visualization', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.colorbar(scatter, label='Cluster', ax=ax)
plt.tight_layout()
plt.savefig('../results/clustering_pca_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ PCA visualization complete")

In [ ]:
# Analyze cluster characteristics
print("\nCluster Characteristics:")

for cluster_id in range(optimal_k):
    cluster_data = df_master[df_master['Cluster'] == cluster_id]
    print(f"\n--- Cluster {cluster_id} ({len(cluster_data)} states) ---")
    print(f"  Mean Disease Rate: {cluster_data['MeanDiseaseRate'].mean():.2f}")
    if 'Total_PerCapita_2015_2020' in cluster_data.columns:
        print(f"  Mean Per Capita Spending: ${cluster_data['Total_PerCapita_2015_2020'].mean():.2f}")
    if 'Physicians_Per100k' in cluster_data.columns:
        print(f"  Mean Physicians per 100k: {cluster_data['Physicians_Per100k'].mean():.2f}")
    print(f"  States: {', '.join(cluster_data['State'].head(5).tolist())}...")

print("\n✓ Cluster analysis complete")

**✓ Commit 17 Complete:** K-means clustering with elbow and silhouette analysis.

## 19. Advanced Model Validation - Cross-Validation

**Extra Analysis (4-person team)**

Implementing k-fold cross-validation to ensure model robustness.

In [ ]:
# Perform 5-Fold Cross-Validation on all models
from sklearn.model_selection import cross_val_score

print("Performing 5-Fold Cross-Validation...")
print("="*60)

models_cv = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, max_depth=5)
}

cv_results = {}

for name, model in models_cv.items():
    print(f"\n{name}:")
    # R² scores
    cv_scores_r2 = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    # Negative MSE (we'll convert to positive RMSE)
    cv_scores_rmse = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
    cv_scores_rmse = np.sqrt(-cv_scores_rmse)
    
    cv_results[name] = {
        'R²_mean': cv_scores_r2.mean(),
        'R²_std': cv_scores_r2.std(),
        'RMSE_mean': cv_scores_rmse.mean(),
        'RMSE_std': cv_scores_rmse.std()
    }
    
    print(f"  R² Score: {cv_scores_r2.mean():.4f} (+/- {cv_scores_r2.std():.4f})")
    print(f"  RMSE: {cv_scores_rmse.mean():.4f} (+/- {cv_scores_rmse.std():.4f})")

print("\n✓ Cross-validation complete")

In [ ]:
# Visualize cross-validation results
cv_df = pd.DataFrame(cv_results).T

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# R² Scores
axes[0].bar(cv_df.index, cv_df['R²_mean'], yerr=cv_df['R²_std'], 
            capsize=5, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('R² Score', fontsize=12, fontweight='bold')
axes[0].set_title('Cross-Validation R² Scores (5-Fold)', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# RMSE
axes[1].bar(cv_df.index, cv_df['RMSE_mean'], yerr=cv_df['RMSE_std'], 
            capsize=5, alpha=0.7, color='orange', edgecolor='black')
axes[1].set_ylabel('RMSE', fontsize=12, fontweight='bold')
axes[1].set_title('Cross-Validation RMSE (5-Fold)', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/ml_cross_validation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Cross-validation visualization complete")

**✓ Commit 18 Complete:** Cross-validation analysis implemented.

## 20. Regional Analysis - Geographic Patterns

**Extra Analysis (4-person team)**

Analyzing chronic disease patterns by U.S. geographic regions.

In [ ]:
# Define U.S. regions
region_mapping = {
    'Northeast': ['Connecticut', 'Maine', 'Massachusetts', 'New Hampshire', 'Rhode Island', 
                  'Vermont', 'New Jersey', 'New York', 'Pennsylvania'],
    'Midwest': ['Illinois', 'Indiana', 'Michigan', 'Ohio', 'Wisconsin', 'Iowa', 'Kansas', 
                'Minnesota', 'Missouri', 'Nebraska', 'North Dakota', 'South Dakota'],
    'South': ['Delaware', 'Florida', 'Georgia', 'Maryland', 'North Carolina', 'South Carolina', 
              'Virginia', 'West Virginia', 'Alabama', 'Kentucky', 'Mississippi', 'Tennessee', 
              'Arkansas', 'Louisiana', 'Oklahoma', 'Texas'],
    'West': ['Arizona', 'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Utah', 
             'Wyoming', 'Alaska', 'California', 'Hawaii', 'Oregon', 'Washington']
}

# Map states to regions
df_master['Region'] = df_master['State'].apply(
    lambda x: next((region for region, states in region_mapping.items() if x in states), 'Unknown')
)

print("Regional distribution:")
print(df_master['Region'].value_counts())
print(f"\n✓ Regional mapping complete")

In [ ]:
# Regional comparison of disease burden
regional_stats = df_master.groupby('Region').agg({
    'MeanDiseaseRate': ['mean', 'std', 'min', 'max'],
    'Total_PerCapita_2015_2020': 'mean',
    'Physicians_Per100k': 'mean',
    'Hospitals_Per100k': 'mean',
    'Prevalence': 'mean',
    'PM25': 'mean'
}).round(2)

print("\nRegional Statistics:")
print(regional_stats)

# Visualize regional disease burden
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Mean Disease Rate by Region
regional_disease = df_master.groupby('Region')['MeanDiseaseRate'].mean().sort_values()
axes[0, 0].barh(regional_disease.index, regional_disease.values, color='steelblue', edgecolor='black')
axes[0, 0].set_xlabel('Mean Disease Rate', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Mean Disease Rate by Region', fontsize=12, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# Plot 2: Per Capita Spending by Region
if 'Total_PerCapita_2015_2020' in df_master.columns:
    regional_spending = df_master.groupby('Region')['Total_PerCapita_2015_2020'].mean().sort_values()
    axes[0, 1].barh(regional_spending.index, regional_spending.values, color='orange', edgecolor='black')
    axes[0, 1].set_xlabel('Per Capita Spending', fontsize=11, fontweight='bold')
    axes[0, 1].set_title('Healthcare Spending by Region', fontsize=12, fontweight='bold')
    axes[0, 1].grid(axis='x', alpha=0.3)

# Plot 3: Physicians per 100k by Region
if 'Physicians_Per100k' in df_master.columns:
    regional_physicians = df_master.groupby('Region')['Physicians_Per100k'].mean().sort_values()
    axes[1, 0].barh(regional_physicians.index, regional_physicians.values, color='green', edgecolor='black')
    axes[1, 0].set_xlabel('Physicians per 100k', fontsize=11, fontweight='bold')
    axes[1, 0].set_title('Physician Density by Region', fontsize=12, fontweight='bold')
    axes[1, 0].grid(axis='x', alpha=0.3)

# Plot 4: Physical Inactivity by Region
if 'Prevalence' in df_master.columns:
    regional_inactivity = df_master.groupby('Region')['Prevalence'].mean().sort_values()
    axes[1, 1].barh(regional_inactivity.index, regional_inactivity.values, color='red', edgecolor='black')
    axes[1, 1].set_xlabel('Inactivity Prevalence (%)', fontsize=11, fontweight='bold')
    axes[1, 1].set_title('Physical Inactivity by Region', fontsize=12, fontweight='bold')
    axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/regional_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Regional analysis visualizations complete")

**✓ Regional analysis complete.**

## 21. Summary and Conclusions

Comprehensive analysis of chronic disease disparities across U.S. states.

In [ ]:
# Final Summary
print("="*80)
print("PROJECT SUMMARY: CHRONIC DISEASE DISPARITIES ANALYSIS")
print("="*80)

print("\n✓ DATASETS INTEGRATED: 8")
print("  - CDC Chronic Disease Surveillance")
print("  - Healthcare Expenditure (2015-2020)")
print("  - Per Capita Healthcare Spending")
print("  - Physicians by State")
print("  - Hospitals by State")
print("  - Physical Inactivity Prevalence")
print("  - PM2.5 Air Pollution")
print("  - Population Data (2020 Census)")

print(f"\n✓ TOTAL STATES ANALYZED: {len(df_master)}")
print(f"\n✓ FEATURES ENGINEERED: {len(feature_cols)}")
print(f"  Features: {', '.join(feature_cols)}")

print("\n✓ ANALYSES COMPLETED:")
print("  - Exploratory Data Analysis (EDA)")
print("  - Research Question Q1: Healthcare Spending Efficiency")
print("  - Research Question Q2: Workforce Composition Impact")
print("  - Research Question Q3: Environmental & Behavioral Factors")
print("  - Regional Analysis (4 U.S. Regions)")

print("\n✓ MACHINE LEARNING MODELS:")
print("  - Linear Regression")
print("  - Random Forest Regressor")
print("  - Gradient Boosting Regressor")
print("  - XGBoost Regressor")
print("  - 5-Fold Cross-Validation")

print("\n✓ UNSUPERVISED LEARNING:")
print("  - K-Means Clustering (with Elbow & Silhouette Analysis)")
print("  - PCA Visualization")
print(f"  - Optimal Clusters: {optimal_k}")

print("\n✓ KEY FINDINGS:")
if 'Total_PerCapita_2015_2020' in df_master.columns:
    spending_corr = df_master['Total_PerCapita_2015_2020'].corr(df_master['MeanDiseaseRate'])
    print(f"  - Spending vs Disease Correlation: {spending_corr:.3f}")
if 'Physicians_Per100k' in df_master.columns:
    phys_corr = df_master['Physicians_Per100k'].corr(df_master['MeanDiseaseRate'])
    print(f"  - Physician Density vs Disease Correlation: {phys_corr:.3f}")
if 'Prevalence' in df_master.columns:
    inact_corr = df_master['Prevalence'].corr(df_master['MeanDiseaseRate'])
    print(f"  - Physical Inactivity vs Disease Correlation: {inact_corr:.3f}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE - ALL 4 TEAM MEMBERS CONTRIBUTED")
print("="*80)